In [4]:
from pathlib import Path
import pickle
import math

In [15]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from qdrant_client import QdrantClient

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qdrant_client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "renovation_docs"

In [2]:
with Path("../data/cleaned/embedded_chunks.pkl").open("rb") as f:
    embedded_chunks = pickle.load(f)

In [3]:
print(len(embedded_chunks))
print(len(embedded_chunks[0].embedding))
print(type(embedded_chunks[0].embedding[0]))
print(embedded_chunks[0].metadata)
print(embedded_chunks[0].text[:300])

1016
1536
<class 'float'>
{'url': 'https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/', 'title': 'Как выбрать подрядчика для ремонта квартиры: советы и рекомендации - Ремонт квартир', 'h1': 'Как выбрать подрядчика для ремонта квартиры: советы и рекомендации', 'headings': ['Как выбрать подрядчика для ремонта квартиры: советы и рекомендации'], 'source': 'sk-family.ru', 'text_length': 2768, 'chunk_index': 0, 'chunk_total': 2}
# Как выбрать подрядчика для ремонта квартиры: советы и рекомендации

Ремонт квартиры — это важный шаг, который требует внимательного подхода. Подрядчик играет ключевую роль в этом процессе. Как же выбрать надежного специалиста? Вот несколько советов от компании СК Family.

1. Определите свои потреб


In [5]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

In [7]:
def embed_query(query: str, model: str = "text-embedding-3-small") -> list[float]:
    response = client.embeddings.create(
        model=model,
        input=query,
    )
    return response.data[0].embedding

In [8]:
def search_chunks(query: str, top_k: int = 5):
    query_embedding = embed_query(query)

    scored_chunks = []

    for chunk in embedded_chunks:
        score = cosine_similarity(query_embedding, chunk.embedding)
        scored_chunks.append((score, chunk))

    scored_chunks.sort(key=lambda item: item[0], reverse=True)

    return scored_chunks[:top_k]

In [9]:
def print_results(results):
    for index, (score, chunk) in enumerate(results, start=1):
        print(f"RESULT #{index}")
        print(f"score: {score:.4f}")
        print(f"url: {chunk.metadata.get('url')}")
        print(f"title: {chunk.metadata.get('title')}")
        print(f"chunk_index: {chunk.metadata.get('chunk_index')}")
        print()
        print(chunk.text[:800])
        print("\n" + "=" * 80 + "\n")

In [10]:
results = search_chunks("как выбрать подрядчика для ремонта", top_k=5)
print_results(results)

RESULT #1
score: 0.7313
url: https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/
title: Как выбрать подрядчика для ремонта квартиры: советы и рекомендации - Ремонт квартир
chunk_index: 0

# Как выбрать подрядчика для ремонта квартиры: советы и рекомендации

Ремонт квартиры — это важный шаг, который требует внимательного подхода. Подрядчик играет ключевую роль в этом процессе. Как же выбрать надежного специалиста? Вот несколько советов от компании СК Family.

1. Определите свои потребности

Перед тем как искать подрядчика, четко определите, какие именно работы необходимо выполнить. Это могут быть косметические изменения, капитальный ремонт или перепланировка. Понимание своих потребностей поможет вам выбрать специалиста с нужным опытом.

2. Исследуйте рынок

Составьте список потенциальных подрядчиков. Используйте интернет, отзывы друзей и знакомых. Обратите внимание на компании с хорошей репутацией, которые работают на рынке не первый год.

3. Проверьте лицензии и сертификаты


RESU

In [11]:
results = search_chunks("сколько стоит демонтаж плитки", top_k=5)
print_results(results)

RESULT #1
score: 0.6520
url: https://sk-family.ru/db.json#demontazh_plitki
title: Демонтаж плитки
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плитки
Единица измерения: кв.м
Цена: 180.0


RESULT #2
score: 0.6457
url: https://sk-family.ru/db.json#snyatie_polistirolnyh_plitok
title: Снятие полистирольных плиток
chunk_index: 0

Категория: Демонтажные работы
Раздел: Потолки
Услуга: Снятие полистирольных плиток
Единица измерения: кв.м
Цена: 200.0


RESULT #3
score: 0.6273
url: https://sk-family.ru/db.json#demontazh_plitochnogo_kleya
title: Демонтаж плиточного клея
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плиточного клея
Единица измерения: кв.м
Цена: 320.0


RESULT #4
score: 0.6182
url: https://sk-family.ru/db.json#demontazh_plintusa_iz_plitki
title: Демонтаж плинтуса из плитки
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плинтуса из плитки
Единица измерения: м/п
Цена: 160.0


RESULT #5
score: 0.

In [12]:
results = search_chunks("дизайнерский ремонт квартиры под ключ", top_k=5)
print_results(results)

RESULT #1
score: 0.6613
url: https://sk-family.ru/dizaynerskiy-remont-kvartir-pod-kluch/
title: Дизайнерский ремонт квартир в Москве под ключ от СК Family
chunk_index: 0

# Дизайнерский ремонт квартир под ключ в Москве от СК Family

Сделаем дизайнерский ремонт квартиры под ключ в Москве под ключ недорого – без задержек и переплат!

Официальный договор

Фиксированная смета

Гарантия до 3 лет

Оплата по факту

## Видео обзоры

## Ответьте на 5 вопросов, и получите расчет стоимости дизайнерского ремонта квартиры под ключ

Свяжемся удобным способом в WhatsApp, Telegram или по телефону — без спама и лишних звонков

## Посмотрите последние наши работы

### Дизайнерский ремонт квартиры под ключ в Москве от СК Family

Ищете идеальное решение для вашего жилья? Дизайнерский ремонт квартиры под ключ в Москве от компании СК Family — это ваш шанс преобразить пространство и создать атмосферу, о которой вы всегда мечтали. Мы предлагаем комплексные услуги по ремонту, начин


RESULT #2
score: 0.6498
ur

# Qdrant

In [19]:
def search_qdrant(query: str, top_k: int = 5):
    query_vector = embed_query(query)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return response.points

In [20]:
def print_qdrant_results(results):
    for index, point in enumerate(results, start=1):
        payload = point.payload or {}

        print(f"RESULT #{index}")
        print(f"score: {point.score:.4f}")
        print(f"url: {payload.get('url')}")
        print(f"title: {payload.get('title')}")
        print(f"chunk_index: {payload.get('chunk_index')}")
        print()
        print((payload.get("text") or "")[:800])
        print("\n" + "=" * 80 + "\n")

In [21]:
results = search_qdrant("как выбрать подрядчика для ремонта", top_k=5)
print_qdrant_results(results)

RESULT #1
score: 0.7313
url: https://sk-family.ru/2025/01/28/kak-vibrat-podryadchika/
title: Как выбрать подрядчика для ремонта квартиры: советы и рекомендации - Ремонт квартир
chunk_index: 0

# Как выбрать подрядчика для ремонта квартиры: советы и рекомендации

Ремонт квартиры — это важный шаг, который требует внимательного подхода. Подрядчик играет ключевую роль в этом процессе. Как же выбрать надежного специалиста? Вот несколько советов от компании СК Family.

1. Определите свои потребности

Перед тем как искать подрядчика, четко определите, какие именно работы необходимо выполнить. Это могут быть косметические изменения, капитальный ремонт или перепланировка. Понимание своих потребностей поможет вам выбрать специалиста с нужным опытом.

2. Исследуйте рынок

Составьте список потенциальных подрядчиков. Используйте интернет, отзывы друзей и знакомых. Обратите внимание на компании с хорошей репутацией, которые работают на рынке не первый год.

3. Проверьте лицензии и сертификаты


RESU

In [22]:
results = search_qdrant("сколько стоит демонтаж плитки", top_k=5)
print_qdrant_results(results)

RESULT #1
score: 0.6521
url: https://sk-family.ru/db.json#demontazh_plitki
title: Демонтаж плитки
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плитки
Единица измерения: кв.м
Цена: 180.0


RESULT #2
score: 0.6456
url: https://sk-family.ru/db.json#snyatie_polistirolnyh_plitok
title: Снятие полистирольных плиток
chunk_index: 0

Категория: Демонтажные работы
Раздел: Потолки
Услуга: Снятие полистирольных плиток
Единица измерения: кв.м
Цена: 200.0


RESULT #3
score: 0.6274
url: https://sk-family.ru/db.json#demontazh_plitochnogo_kleya
title: Демонтаж плиточного клея
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плиточного клея
Единица измерения: кв.м
Цена: 320.0


RESULT #4
score: 0.6182
url: https://sk-family.ru/db.json#demontazh_plintusa_iz_plitki
title: Демонтаж плинтуса из плитки
chunk_index: 0

Категория: Демонтажные работы
Раздел: Полы
Услуга: Демонтаж плинтуса из плитки
Единица измерения: м/п
Цена: 160.0


RESULT #5
score: 0.

In [23]:
results = search_qdrant("дизайнерский ремонт квартиры под ключ", top_k=5)
print_qdrant_results(results)

RESULT #1
score: 0.6613
url: https://sk-family.ru/dizaynerskiy-remont-kvartir-pod-kluch/
title: Дизайнерский ремонт квартир в Москве под ключ от СК Family
chunk_index: 0

# Дизайнерский ремонт квартир под ключ в Москве от СК Family

Сделаем дизайнерский ремонт квартиры под ключ в Москве под ключ недорого – без задержек и переплат!

Официальный договор

Фиксированная смета

Гарантия до 3 лет

Оплата по факту

## Видео обзоры

## Ответьте на 5 вопросов, и получите расчет стоимости дизайнерского ремонта квартиры под ключ

Свяжемся удобным способом в WhatsApp, Telegram или по телефону — без спама и лишних звонков

## Посмотрите последние наши работы

### Дизайнерский ремонт квартиры под ключ в Москве от СК Family

Ищете идеальное решение для вашего жилья? Дизайнерский ремонт квартиры под ключ в Москве от компании СК Family — это ваш шанс преобразить пространство и создать атмосферу, о которой вы всегда мечтали. Мы предлагаем комплексные услуги по ремонту, начин


RESULT #2
score: 0.6498
ur